In [2]:
# Import libraries
import matplotlib.pyplot as plt
import koreanize_matplotlib

import numpy as np
import pandas as pd

from urllib.request import urlopen
import json


import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import folium


## 1. 전처리된 2025년 생활인구 데이터 불러오기 

In [3]:
SPOP_2025_ADM = pd.read_csv(f'data\SPOP_2025_ADM.csv')
SPOP_2025_ADM

<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\leeja\AppData\Local\Temp\ipykernel_29140\3003728435.py:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  SPOP_2025_ADM = pd.read_csv(f'data\SPOP_2025_ADM.csv')


,YMD,TT,H_DNG_CD,SPOP,M00,M10,M20,M30,M40,M50,...,F60,F70,ADMI_CD,SIDO_NM,SGG_NM,ADMI_NM,FULL_NM,HDAYS,WEEKEND,WORKDAY
0,20250101,0,11110515,14205.95,70.38,669.04,851.96,1021.66,1032.29,1131.88,...,1049.37,1214.13,11110515,서울특별시,종로구,청운효자동,서울특별시 종로구 청운효자동,1,0,0
1,20250101,0,11110530,15529.34,29.76,483.48,1191.52,1383.22,1179.69,1260.15,...,1115.81,1122.42,11110530,서울특별시,종로구,사직동,서울특별시 종로구 사직동,1,0,0
2,20250101,0,11110540,3296.03,6.22,114.46,299.31,228.36,238.77,261.97,...,286.65,323.15,11110540,서울특별시,종로구,삼청동,서울특별시 종로구 삼청동,1,0,0
3,20250101,0,11110550,11377.86,70.98,511.48,655.00,724.95,805.07,989.88,...,985.28,1033.39,11110550,서울특별시,종로구,부암동,서울특별시 종로구 부암동,1,0,0
4,20250101,0,11110560,19725.03,108.25,789.01,1130.13,1273.91,1310.67,1670.46,...,1859.78,1606.73,11110560,서울특별시,종로구,평창동,서울특별시 종로구 평창동,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3731755,20251231,19,11230536,41645.33,111.51,860.29,2556.25,3803.83,3484.63,3891.25,...,3375.74,2975.48,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동,0,0,1
3731756,20251231,20,11230536,41793.42,111.40,878.63,2663.29,3991.45,3503.55,3845.54,...,3308.61,2921.69,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동,0,0,1
3731757,20251231,21,11230536,39425.33,105.04,821.77,2491.18,3914.05,3263.19,3570.64,...,3006.68,2719.46,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동,0,0,1
3731758,20251231,22,11230536,40861.42,103.68,847.92,2661.42,4094.05,3458.72,3639.98,...,3067.54,2867.92,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동,0,0,1


## 2. 2025년 행정동별/시간대별 연앙 인구 산출

In [5]:
SPOP_2025_ADM_age_mid=SPOP_2025_ADM.groupby(['SIDO_NM','SGG_NM','ADMI_NM', 'FULL_NM', 'ADMI_CD', 'TT'])[['M00','M10','M20','M30','M40','M50','M60','M70',
                                                                            'F00','F10','F20','F30','F40','F50','F60','F70']].agg(['mean'])
SPOP_2025_ADM_age_mid=SPOP_2025_ADM_age_mid.droplevel(axis=1,level=1).reset_index()
SPOP_2025_ADM_age_mid.head()

,SIDO_NM,SGG_NM,ADMI_NM,FULL_NM,ADMI_CD,TT,M00,M10,M20,M30,...,M60,M70,F00,F10,F20,F30,F40,F50,F60,F70
0,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,0,275.033288,1928.046438,1297.987288,1593.549507,...,1964.523123,1627.471562,422.377479,2084.254192,1380.570137,2025.721918,3183.829863,2440.103753,2166.950630,1801.820384
1,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,1,275.615151,1940.597781,1319.072822,1610.459918,...,1969.981945,1630.532082,422.896055,2097.691945,1394.593781,2037.670932,3195.447397,2446.488192,2169.275370,1803.322055
2,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,2,276.188356,1949.872548,1340.163288,1629.173315,...,1981.427973,1639.624740,423.404795,2106.734959,1406.251644,2047.228027,3207.951260,2456.617096,2178.726658,1811.302055
3,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,3,276.317397,1952.184959,1352.184247,1639.017808,...,1987.225671,1643.018247,423.157014,2109.047096,1411.633041,2050.437918,3210.776740,2458.554685,2180.731781,1811.729644
4,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,4,276.085562,1951.754630,1356.287507,1645.887644,...,1987.961014,1644.108301,422.731205,2108.539452,1411.571644,2048.960575,3208.770849,2454.978959,2176.889562,1807.057589


In [6]:
# csv 파일로 저장
SPOP_2025_ADM_age_mid.to_csv('data/SPOP_2025_ADM_age_mid.csv', index=False, encoding='utf-8-sig') 

In [78]:
# 다시 불러오기
SPOP_2025_ADM_age_mid = pd.read_csv(f'data\SPOP_2025_ADM_age_mid.csv')
SPOP_2025_ADM_age_mid

<>:2: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:2: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\leeja\AppData\Local\Temp\ipykernel_29140\778287588.py:2: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  SPOP_2025_ADM_age_mid = pd.read_csv(f'data\SPOP_2025_ADM_age_mid.csv')


,SIDO_NM,SGG_NM,ADMI_NM,FULL_NM,ADMI_CD,TT,M00,M10,M20,M30,...,M60,M70,F00,F10,F20,F30,F40,F50,F60,F70
0,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,0,275.033288,1928.046438,1297.987288,1593.549507,...,1964.523123,1627.471562,422.377479,2084.254192,1380.570137,2025.721918,3183.829863,2440.103753,2166.950630,1801.820384
1,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,1,275.615151,1940.597781,1319.072822,1610.459918,...,1969.981945,1630.532082,422.896055,2097.691945,1394.593781,2037.670932,3195.447397,2446.488192,2169.275370,1803.322055
2,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,2,276.188356,1949.872548,1340.163288,1629.173315,...,1981.427973,1639.624740,423.404795,2106.734959,1406.251644,2047.228027,3207.951260,2456.617096,2178.726658,1811.302055
3,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,3,276.317397,1952.184959,1352.184247,1639.017808,...,1987.225671,1643.018247,423.157014,2109.047096,1411.633041,2050.437918,3210.776740,2458.554685,2180.731781,1811.729644
4,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,4,276.085562,1951.754630,1356.287507,1645.887644,...,1987.961014,1644.108301,422.731205,2108.539452,1411.571644,2048.960575,3208.770849,2454.978959,2176.889562,1807.057589
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10219,서울특별시,중랑구,중화2동,서울특별시 중랑구 중화2동,11260610,19,108.138466,529.099890,1187.381973,1733.445425,...,2491.124822,1874.828219,105.159123,396.994000,1223.293041,1471.666658,1336.811452,2004.004356,2253.088740,2375.575836
10220,서울특별시,중랑구,중화2동,서울특별시 중랑구 중화2동,11260610,20,110.938603,529.986219,1227.650274,1824.468795,...,2520.741589,1882.101671,108.469014,409.912932,1319.511863,1558.124082,1368.035014,2042.744137,2275.243562,2363.277315
10221,서울특별시,중랑구,중화2동,서울특별시 중랑구 중화2동,11260610,21,113.431781,537.452055,1268.997151,1895.303836,...,2530.872630,1882.756192,110.993863,418.642904,1398.200986,1625.774767,1397.923918,2081.999507,2289.392795,2351.577397
10222,서울특별시,중랑구,중화2동,서울특별시 중랑구 중화2동,11260610,22,114.142110,548.062521,1319.328000,1966.108466,...,2531.704795,1881.864986,112.918904,438.757151,1479.279123,1677.110630,1437.359288,2115.595644,2307.081452,2344.515753


## 3. 2025년 연앙 오후 2시 관악구 대학동 인구 피라미드 시각화

In [79]:
SPOP_2025_ADM_age_mid.loc[SPOP_2025_ADM_age_mid['SGG_NM'] == '관악구', 'ADMI_NM'].unique()

<StringArray>
['낙성대동',  '난곡동',  '난향동',  '남현동',  '대학동',  '미성동', '보라매동',  '삼성동',  '서림동',
  '서원동',  '성현동',  '신림동',  '신사동',  '신원동',  '은천동',  '인헌동',  '조원동',  '중앙동',
  '청룡동',  '청림동',  '행운동']
Length: 21, dtype: str

In [80]:
SPOP_2022_daehak = SPOP_2025_ADM_age_mid[SPOP_2025_ADM_age_mid['ADMI_NM']=='대학동']
SPOP_2022_daehak = SPOP_2022_daehak[SPOP_2022_daehak['TT']==14]

SPOP_2022_daehak

,SIDO_NM,SGG_NM,ADMI_NM,FULL_NM,ADMI_CD,TT,M00,M10,M20,M30,...,M60,M70,F00,F10,F20,F30,F40,F50,F60,F70
1886,서울특별시,관악구,대학동,서울특별시 관악구 대학동,11620735,14,65.772603,1678.414055,9331.658356,5323.379041,...,2255.150521,1567.738959,55.384137,1208.124,6026.556384,2971.259068,2222.406466,1922.245644,1712.87463,1679.030822


In [81]:
pyramid_df

,연령대,남성,여성
0,0~9세,-65.772603,55.384137
1,10~19세,-1678.414055,1208.124000
2,20~29세,-9331.658356,6026.556384
3,30~39세,-5323.379041,2971.259068
4,40~49세,-3156.965726,2222.406466
5,50~59세,-3365.771699,1922.245644
6,60~69세,-2255.150521,1712.874630
7,70세 이상,-1567.738959,1679.030822


In [82]:
import pandas as pd
import plotly.express as px

age_labels = ['0~9세', '10~19세', '20~29세', '30~39세',
              '40~49세', '50~59세', '60~69세', '70세 이상']

male_cols = [f'M{age:02d}' for age in range(0, 80, 10)]
female_cols = [f'F{age:02d}' for age in range(0, 80, 10)]

row = SPOP_2022_daehak.iloc[0]

# wide → long 형식으로 변환
pyramid_df = pd.DataFrame({
    '연령대': age_labels,
    '남성': [-row[col] for col in male_cols],  # 왼쪽 표시
    '여성': [row[col] for col in female_cols],
})

pyramid_long = pyramid_df.melt(
    id_vars='연령대',
    value_vars=['남성', '여성'],
    var_name='성별',
    value_name='생활인구'
)

fig = px.bar(
    pyramid_long,
    x='생활인구',
    y='연령대',
    color='성별',
    orientation='h',
    barmode='relative',
    category_orders={'연령대': age_labels[::-1]},
    color_discrete_map={'남성': '#4C78A8', '여성': '#E45756'},
    title=f"{row['FULL_NM']} {row['TT']}시 생활인구 피라미드",
    labels={'생활인구': '생활인구 (명)', '연령대': '연령대'},
    template='simple_white'
)

fig.update_xaxes(
    tickformat=',.0f',
    zeroline=True,
    zerolinecolor='gray'
)

fig.show()
fig.write_html("html/2_1.html")


## 4. 행정동별 생활인구 피라미드 함수화

In [83]:
import pandas as pd
import plotly.express as px

def plot_population_pyramid(df, gu_name, dong_name, hour=14, title=None):
    """
    선택한 행정동과 시간대의 생활인구 피라미드를 그립니다.

    Parameters
    ----------
    df : pandas.DataFrame
        SPOP_2025_ADM_age_mid 형식의 데이터프레임
    gu_name : str
        자치구명. 예: '관악구'
    dong_name : str
        행정동명. 예: '대학동'
    hour : int, default=14
        조회 시간대 (0~23시)
    title : str, optional
        그래프 제목. 지정하지 않으면 행정동명과 시간대를 사용합니다.

    Returns
    -------
    plotly.graph_objects.Figure
    """
    # 구·동·시간대 필터링
    selected = df.loc[
        (df['SGG_NM'] == gu_name) &
        (df['ADMI_NM'] == dong_name) &
        (df['TT'] == hour)
    ].copy()

    if selected.empty:
        raise ValueError(
            f"'{gu_name} {dong_name}'의 {hour}시 데이터가 없습니다."
        )

    if len(selected) > 1:
        raise ValueError(
            "조건에 해당하는 데이터가 여러 행입니다. 데이터 중복 여부를 확인하세요."
        )

    age_labels = [
        '0~9세', '10~19세', '20~29세', '30~39세',
        '40~49세', '50~59세', '60~69세', '70세 이상'
    ]
    male_cols = ['M00', 'M10', 'M20', 'M30', 'M40', 'M50', 'M60', 'M70']
    female_cols = ['F00', 'F10', 'F20', 'F30', 'F40', 'F50', 'F60', 'F70']

    row = selected.iloc[0]

    pyramid_wide = pd.DataFrame({
        '연령대': age_labels,
        '남성': [-row[col] for col in male_cols],  # 왼쪽 방향
        '여성': [row[col] for col in female_cols],
    })

    pyramid_long = pyramid_wide.melt(
        id_vars='연령대',
        value_vars=['남성', '여성'],
        var_name='성별',
        value_name='생활인구'
    )

    if title is None:
        title = f'{gu_name} {dong_name} {hour}시 생활인구 피라미드'

    fig = px.bar(
        pyramid_long,
        x='생활인구',
        y='연령대',
        color='성별',
        orientation='h',
        barmode='relative',
        category_orders={'연령대': age_labels[::-1]},
        color_discrete_map={'남성': '#4C78A8', '여성': '#E45756'},
        title=title,
        labels={'생활인구': '생활인구 (명)', '연령대': '연령대'},
        template='simple_white'
    )

    fig.update_xaxes(
        tickformat=',.0f',
        zeroline=True,
        zerolinecolor='gray'
    )

    return fig

In [84]:
fig = plot_population_pyramid(
    SPOP_2025_ADM_age_mid,
    gu_name='관악구',
    dong_name='난곡동',
    hour=14
)

fig.show()
fig.write_html("html/2_2.html")


## 5. 구별 생활인구 피라미드 함수화

In [85]:
import pandas as pd
import plotly.express as px

def plot_gu_population_pyramids(df, gu_name, hour=14, facet_col_wrap=4):
    """
    선택한 구의 행정동별·시간대별 생활인구 피라미드를 그립니다.

    Parameters
    ----------
    df : pandas.DataFrame
        SPOP_2025_ADM_age_mid 형식의 데이터프레임
    gu_name : str
        자치구명. 예: '관악구'
    hour : int, default=14
        조회 시간대 (0~23시)
    facet_col_wrap : int, default=4
        한 줄에 표시할 행정동 피라미드 개수

    Returns
    -------
    plotly.graph_objects.Figure
    """
    # 선택한 구와 시간대 필터링
    selected = df.loc[
        (df['SGG_NM'] == gu_name) &
        (df['TT'] == hour)
    ].copy()

    if selected.empty:
        raise ValueError(
            f"'{gu_name}'의 {hour}시 데이터가 없습니다."
        )

    age_labels = [
        '0~9세', '10~19세', '20~29세', '30~39세',
        '40~49세', '50~59세', '60~69세', '70세 이상'
    ]
    male_cols = ['M00', 'M10', 'M20', 'M30', 'M40', 'M50', 'M60', 'M70']
    female_cols = ['F00', 'F10', 'F20', 'F30', 'F40', 'F50', 'F60', 'F70']

    # 행정동별 wide → long 변환
    records = []

    for _, row in selected.iterrows():
        for age_label, male_col, female_col in zip(
            age_labels, male_cols, female_cols
        ):
            records.append({
                '행정동': row['ADMI_NM'],
                '연령대': age_label,
                '성별': '남성',
                '생활인구': -row[male_col],  # 왼쪽 방향
                '실제인구': row[male_col],
            })
            records.append({
                '행정동': row['ADMI_NM'],
                '연령대': age_label,
                '성별': '여성',
                '생활인구': row[female_col],
                '실제인구': row[female_col],
            })

    pyramid_long = pd.DataFrame(records)
    max_population = pyramid_long['실제인구'].max()

    fig = px.bar(
        pyramid_long,
        x='생활인구',
        y='연령대',
        color='성별',
        facet_col='행정동',
        facet_col_wrap=facet_col_wrap,
        orientation='h',
        barmode='relative',
        category_orders={'연령대': age_labels[::-1]},
        color_discrete_map={'남성': '#4C78A8', '여성': '#E45756'},
        hover_data={
            '생활인구': False,
            '실제인구': ':,.0f'
        },
        labels={
            '실제인구': '생활인구',
            '연령대': '연령대',
            '성별': '성별'
        },
        title=f'{gu_name} 행정동별 {hour}시 생활인구 피라미드',
        template='simple_white',
        height=((len(selected) - 1) // facet_col_wrap + 1) * 300
    )

    fig.update_xaxes(
        range=[-max_population * 1.15, max_population * 1.15],
        tickformat=',.0f',
        zeroline=True,
        zerolinecolor='gray'
    )
    
    # 0~9세부터 70세 이상까지 8개 연령대를 처음부터 모두 표시
    fig.update_yaxes(
        categoryorder='array',
        categoryarray=age_labels[::1],
        range=[-0.5, len(age_labels) - 0.5],
        autorange=False
    )

    fig.for_each_annotation(
        lambda annotation: annotation.update(
            text=annotation.text.replace('행정동=', '')
        )
    )

    fig.update_layout(legend_title_text='성별')

    return fig

In [86]:
fig = plot_gu_population_pyramids(
    SPOP_2025_ADM_age_mid,
    gu_name='관악구',
    hour=14,
    facet_col_wrap=5
)

fig.show()
fig.write_html("html/2_3.html")


## 6. 생활인구 피라미드가 유사한 지역 찾기

### 인구 피라미드 유사도 지표 비교

행정동별 생활인구 피라미드는 성별·연령대별 인구 구성으로 이루어집니다.  
유사도를 계산할 때는 **전체 인구 규모**, **성별·연령대 구성 비율**, **연령대 간 거리** 중 무엇을 중요하게 볼 것인지에 따라 지표를 선택해야 합니다.

| 지표 | 비교 관점 | 장점 | 한계 | 인구 피라미드 활용 적합성 |
|---|---|---|---|---|
| 코사인 유사도 | 성별·연령대 구성 벡터의 방향 | 계산이 간단하고, 전체 인구 규모가 달라도 구성 비율이 유사하면 높은 값이 나옴 | 20대와 30대의 차이, 20대와 70대의 차이를 동일하게 취급 | 기본 비교 지표로 적합 |
| Jensen–Shannon 거리 | 성별·연령대별 인구 비율 분포 | 확률분포 간 차이를 비교하므로 피라미드의 구성비 차이를 해석하기 쉬움 | 연령대의 순서나 인접성을 반영하지 않음 | 구성비 중심 비교에 적합 |
| Wasserstein 거리 | 연령대 분포가 이동해야 하는 거리 | 20대 중심과 30대 중심은 가깝게, 20대 중심과 70대 중심은 멀게 평가하여 연령대 순서를 반영 | 남성·여성을 분리해 계산하고 결합하는 과정이 필요하며 해석이 상대적으로 복잡함 | 연령구조 형태 비교에 가장 적합 |

#### 1. 코사인 유사도

두 행정동의 성별·연령대별 생활인구 벡터가 이루는 각도를 이용해 유사도를 계산합니다.

$$
\mathrm{Cosine\ Similarity}(A, B)
=
\frac{A \cdot B}
{\lVert A \rVert \times \lVert B \rVert}
$$

- 값이 **1에 가까울수록** 두 인구 피라미드의 성별·연령대 구성이 유사합니다.
- 전체 생활인구 규모보다 성별·연령대 구성의 비율에 초점을 둡니다.
- 다만 20대와 30대의 차이, 20대와 70대의 차이를 동일한 차이로 처리합니다.

#### 2. Jensen–Shannon 거리

두 행정동의 성별·연령대별 생활인구를 전체 생활인구 대비 비율로 변환한 뒤, 두 확률분포의 차이를 계산합니다.

$$
\mathrm{JSD}(P, Q)
=
\sqrt{
\frac{
D_{KL}(P \parallel M)
+
D_{KL}(Q \parallel M)
}{2}
}
$$

$$
M = \frac{P + Q}{2}
$$

- 값이 **0에 가까울수록** 두 행정동의 성별·연령대별 구성비가 유사합니다.
- 생활인구의 절대 규모를 제외하고 구성비만 비교할 때 유용합니다.
- 연령대가 순서형 범주라는 특성은 반영하지 않습니다.

#### 3. Wasserstein 거리

한 행정동의 연령분포를 다른 행정동의 연령분포로 바꾸기 위해 인구를 연령대 사이로 이동시키는 데 필요한 최소 비용을 계산합니다.

- 값이 **0에 가까울수록** 두 연령분포가 유사합니다.
- 인접한 연령대끼리의 차이는 작게, 멀리 떨어진 연령대의 차이는 크게 평가합니다.
- 예를 들어 20대 중심 행정동은 30대 중심 행정동보다 70대 중심 행정동과 더 큰 거리를 가집니다.
- 남성과 여성의 연령분포를 각각 계산한 뒤 평균 또는 가중평균하는 방식을 사용할 수 있습니다.

#### 지표 선택 기준

- 단순하고 직관적인 비교가 필요하면: **코사인 유사도**
- 성별·연령대 구성비 자체를 비교하려면: **Jensen–Shannon 거리**
- 연령대의 인접성까지 반영해 피라미드 형태를 비교하려면: **Wasserstein 거리**

본 분석에서는 코사인 유사도를 기본 지표로 사용하되, 연령구조의 이동까지 고려해야 하는 경우 Wasserstein 거리를 함께 활용할 수 있습니다.

In [116]:
# Cosine Similarity로 가장 유사한 행정동 Top 5를 찾아 생활인구 피라미드로 비교

import numpy as np
import pandas as pd
import plotly.express as px


def cosine_similarity_pyramid(
    df,
    gu_name,
    dong_name,
    hour=14,
    top_n=5,
    facet_col_wrap=3
):
    """
    특정 행정동과 같은 시간대에 생활인구 피라미드가 가장 유사한
    행정동 top_n개를 찾고, 인구 피라미드로 비교합니다.

    Parameters
    ----------
    df : pandas.DataFrame
        SPOP_2025_ADM_age_mid 형식의 데이터프레임
    gu_name : str
        기준 자치구명. 예: '관악구'
    dong_name : str
        기준 행정동명. 예: '대학동'
    hour : int, default=14
        비교 시간대 (0~23시)
    top_n : int, default=5
        반환할 유사 행정동 수
    facet_col_wrap : int, default=3
        한 줄에 표시할 피라미드 수

    Returns
    -------
    fig : plotly.graph_objects.Figure
        기준 행정동과 유사 행정동의 인구 피라미드
    similar_dongs : pandas.DataFrame
        코사인 유사도 상위 행정동 목록
    """

    age_labels = [
        '0~9세', '10~19세', '20~29세', '30~39세',
        '40~49세', '50~59세', '60~69세', '70세 이상'
    ]
    
    male_cols = ['M00', 'M10', 'M20', 'M30', 'M40', 'M50', 'M60', 'M70']
    female_cols = ['F00', 'F10', 'F20', 'F30', 'F40', 'F50', 'F60', 'F70']
    feature_cols = male_cols + female_cols

    # 같은 시간대의 서울 행정동 데이터만 추출
    comparison_df = df.loc[df['TT'] == hour].copy()

    # 기준 행정동 추출
    target = comparison_df.loc[
        (comparison_df['SGG_NM'] == gu_name) &
        (comparison_df['ADMI_NM'] == dong_name)
    ].copy()

    if target.empty:
        raise ValueError(
            f"'{gu_name} {dong_name}'의 {hour}시 데이터가 없습니다."
        )

    if len(target) > 1:
        raise ValueError(
            "기준 행정동 데이터가 여러 행입니다. 데이터 중복 여부를 확인하세요."
        )

    target_code = target.iloc[0]['ADMI_CD']

    # 코사인 유사도 계산
    feature_matrix = comparison_df[feature_cols].fillna(0).to_numpy(dtype=float)
    target_vector = target[feature_cols].fillna(0).to_numpy(dtype=float)[0]

    vector_norms = np.linalg.norm(feature_matrix, axis=1)
    target_norm = np.linalg.norm(target_vector)

    if target_norm == 0:
        raise ValueError("기준 행정동의 생활인구 값이 모두 0입니다.")

    cosine_similarity = (
        feature_matrix @ target_vector
    ) / (vector_norms * target_norm)

    comparison_df['COSINE_SIMILARITY'] = cosine_similarity

    # 기준 행정동을 제외한 상위 유사 행정동 선택
    similar_dongs = (
        comparison_df.loc[comparison_df['ADMI_CD'] != target_code]
        .sort_values('COSINE_SIMILARITY', ascending=False)
        .head(top_n)
        [['SGG_NM', 'ADMI_NM', 'FULL_NM', 'ADMI_CD', 'COSINE_SIMILARITY']]
        .reset_index(drop=True)
    )

    similar_dongs.index = similar_dongs.index + 1
    similar_dongs.index.name = '순위'

    # 기준 행정동 + 유사 행정동을 시각화 대상으로 구성
    selected_codes = [target_code] + similar_dongs['ADMI_CD'].tolist()

    plot_df = comparison_df.loc[
        comparison_df['ADMI_CD'].isin(selected_codes)
    ].copy()

    # facet 제목용 이름 생성
    target_name = f'기준: {gu_name} {dong_name}'

    similarity_map = {
        row['ADMI_CD']: (
            f"{row['SGG_NM']} {row['ADMI_NM']}<br>"
            f"유사도: {row['COSINE_SIMILARITY']:.3f}"
        )
        for _, row in similar_dongs.reset_index().iterrows()
    }

    plot_df['표시명'] = plot_df['ADMI_CD'].map(similarity_map)
    plot_df.loc[plot_df['ADMI_CD'] == target_code, '표시명'] = target_name

    # wide → long 변환
    records = []

    for _, row in plot_df.iterrows():
        for age_label, male_col, female_col in zip(
            age_labels, male_cols, female_cols
        ):
            records.append({
                '행정동': row['표시명'],
                '연령대': age_label,
                '성별': '남성',
                '생활인구': -row[male_col],
                '실제인구': row[male_col],
            })
            records.append({
                '행정동': row['표시명'],
                '연령대': age_label,
                '성별': '여성',
                '생활인구': row[female_col],
                '실제인구': row[female_col],
            })

    pyramid_long = pd.DataFrame(records)
    max_population = pyramid_long['실제인구'].max()

    fig = px.bar(
        pyramid_long,
        x='생활인구',
        y='연령대',
        color='성별',
        facet_col='행정동',
        facet_col_wrap=facet_col_wrap,
        orientation='h',
        barmode='relative',
        category_orders={'연령대': age_labels[::-1]},
        color_discrete_map={'남성': '#4C78A8', '여성': '#E45756'},
        hover_data={
            '생활인구': False,
            '실제인구': ':,.0f'
        },
        labels={
            '실제인구': '생활인구',
            '연령대': '연령대',
            '성별': '성별'
        },
        title=(
            f'{gu_name} {dong_name} · {hour}시 생활인구 피라미드 '
            f'코사인 유사 행정동 Top {top_n}'
        ),
        template='simple_white',
        height=((top_n) // facet_col_wrap + 1) * 450
    )

    fig.update_xaxes(
        range=[-max_population * 1.15, max_population * 1.15],
        tickformat=',.0f',
        zeroline=True,
        zerolinecolor='gray'
    )

    # 처음부터 70세 이상까지 8개 연령대 전체 표시
    fig.update_yaxes(
        categoryorder='array',
        categoryarray=age_labels[::1],
        range=[-0.5, len(age_labels) - 0.5],
        autorange=False
    )

    fig.for_each_annotation(
        lambda annotation: annotation.update(
            text=annotation.text.replace('행정동=', '')
        )
    )

    fig.update_layout(legend_title_text='성별')

    return fig, similar_dongs

In [117]:
# Wasserstein 거리로 가장 유사한 행정동 Top 5를 찾아 생활인구 피라미드로 비교

import numpy as np
import pandas as pd
import plotly.express as px


def wasserstein_distance_pyramid(
    df,
    gu_name,
    dong_name,
    hour=14,
    top_n=5,
    facet_col_wrap=3
):
    """
    기준 행정동과 같은 시간대의 행정동 중,
    Wasserstein 거리가 가장 작은 행정동을 찾아 인구 피라미드로 비교합니다.

    Wasserstein 거리가 작을수록 성별·연령대별 인구 구조가 유사합니다.
    """

    age_labels = [
        '0~9세', '10~19세', '20~29세', '30~39세',
        '40~49세', '50~59세', '60~69세', '70세 이상'
    ]
    male_cols = ['M00', 'M10', 'M20', 'M30', 'M40', 'M50', 'M60', 'M70']
    female_cols = ['F00', 'F10', 'F20', 'F30', 'F40', 'F50', 'F60', 'F70']

    required_cols = [
        'SGG_NM', 'ADMI_NM', 'ADMI_CD', 'TT',
        *male_cols, *female_cols
    ]
    missing_cols = set(required_cols) - set(df.columns)

    if missing_cols:
        raise KeyError(f'필수 열이 없습니다: {sorted(missing_cols)}')

    # 같은 시간대의 전체 행정동 추출
    comparison_df = df.loc[df['TT'] == hour].copy()

    # 기준 행정동 추출
    target = comparison_df.loc[
        (comparison_df['SGG_NM'] == gu_name) &
        (comparison_df['ADMI_NM'] == dong_name)
    ].copy()

    if target.empty:
        raise ValueError(
            f"'{gu_name} {dong_name}'의 {hour}시 데이터가 없습니다."
        )

    if len(target) > 1:
        raise ValueError(
            '기준 행정동 데이터가 여러 행입니다. 중복 여부를 확인하세요.'
        )

    target_code = target.iloc[0]['ADMI_CD']

    def wasserstein_age_distance(counts_a, counts_b):
        """연령대별 인구분포 간 1차원 Wasserstein 거리"""
        counts_a = np.asarray(counts_a, dtype=float)
        counts_b = np.asarray(counts_b, dtype=float)

        if counts_a.sum() == 0 or counts_b.sum() == 0:
            return np.nan

        # 인구 규모를 제외하고 연령대별 구성비만 비교
        p = counts_a / counts_a.sum()
        q = counts_b / counts_b.sum()

        # 연령대 간격을 10년으로 간주
        return np.abs(np.cumsum(p) - np.cumsum(q)).sum() * 10

    target_male = target.iloc[0][male_cols].fillna(0).to_numpy(dtype=float)
    target_female = target.iloc[0][female_cols].fillna(0).to_numpy(dtype=float)

    wasserstein_distances = []

    for _, row in comparison_df.iterrows():
        candidate_male = row[male_cols].fillna(0).to_numpy(dtype=float)
        candidate_female = row[female_cols].fillna(0).to_numpy(dtype=float)

        male_distance = wasserstein_age_distance(
            target_male, candidate_male
        )
        female_distance = wasserstein_age_distance(
            target_female, candidate_female
        )

        # 남녀 연령구조의 거리를 같은 비중으로 평균
        distance = np.nanmean([male_distance, female_distance])
        wasserstein_distances.append(distance)

    comparison_df['WASSERSTEIN_DISTANCE'] = wasserstein_distances

    # 기준 행정동을 제외하고 거리가 작은 순으로 상위 행정동 선택
    similar_dongs = (
        comparison_df.loc[comparison_df['ADMI_CD'] != target_code]
        .sort_values('WASSERSTEIN_DISTANCE', ascending=True)
        .head(top_n)
        [['SGG_NM', 'ADMI_NM', 'FULL_NM', 'ADMI_CD', 'WASSERSTEIN_DISTANCE']]
        .reset_index(drop=True)
    )

    similar_dongs.index = similar_dongs.index + 1
    similar_dongs.index.name = '순위'

    # 기준 행정동과 유사 행정동만 시각화 대상으로 선택
    selected_codes = [target_code] + similar_dongs['ADMI_CD'].tolist()

    plot_df = comparison_df.loc[
        comparison_df['ADMI_CD'].isin(selected_codes)
    ].copy()

    target_label = f'기준: {gu_name} {dong_name}'

    distance_map = {
        row['ADMI_CD']: (
            f"{row['SGG_NM']} {row['ADMI_NM']}<br>"
            f"거리: {row['WASSERSTEIN_DISTANCE']:.3f}"
        )
        for _, row in similar_dongs.reset_index().iterrows()
    }

    plot_df['표시명'] = plot_df['ADMI_CD'].map(distance_map)
    plot_df.loc[
        plot_df['ADMI_CD'] == target_code, '표시명'
    ] = target_label

    display_order = (
        [target_label] +
        [
            distance_map[code]
            for code in similar_dongs['ADMI_CD']
        ]
    )

    # 행정동별 wide → long 변환
    records = []

    for _, row in plot_df.iterrows():
        for age_label, male_col, female_col in zip(
            age_labels, male_cols, female_cols
        ):
            records.append({
                '행정동': row['표시명'],
                '연령대': age_label,
                '성별': '남성',
                '생활인구': -row[male_col],
                '실제인구': row[male_col],
            })
            records.append({
                '행정동': row['표시명'],
                '연령대': age_label,
                '성별': '여성',
                '생활인구': row[female_col],
                '실제인구': row[female_col],
            })

    pyramid_long = pd.DataFrame(records)
    max_population = pyramid_long['실제인구'].max()

    fig = px.bar(
        pyramid_long,
        x='생활인구',
        y='연령대',
        color='성별',
        facet_col='행정동',
        facet_col_wrap=facet_col_wrap,
        orientation='h',
        barmode='relative',
        category_orders={
            '연령대': age_labels[::-1],
            '행정동': display_order
        },
        color_discrete_map={'남성': '#4C78A8', '여성': '#E45756'},
        hover_data={
            '생활인구': False,
            '실제인구': ':,.0f'
        },
        labels={
            '실제인구': '생활인구',
            '연령대': '연령대',
            '성별': '성별'
        },
        title=(
            f'{gu_name} {dong_name} · {hour}시 생활인구 피라미드 '
            f'Wasserstein 거리 기준 유사 행정동 Top {top_n}'
        ),
        template='simple_white',
        height=(
            ((top_n + 1 + facet_col_wrap - 1) // facet_col_wrap) * 450
        )
    )

    fig.update_xaxes(
        range=[-max_population * 1.15, max_population * 1.15],
        tickformat=',.0f',
        zeroline=True,
        zerolinecolor='gray'
    )

    # 모든 연령대가 초기 화면에 표시되도록 고정
    fig.update_yaxes(
        categoryorder='array',
        categoryarray=age_labels[::1],
        range=[-0.5, len(age_labels) - 0.5],
        autorange=False
    )

    fig.for_each_annotation(
        lambda annotation: annotation.update(
            text=annotation.text.replace('행정동=', '')
        )
    )

    fig.update_layout(legend_title_text='성별')

    return fig, similar_dongs

In [118]:
# 서울대학교가 있는 관악구 대학동, 오후 2시 생활인구 피라미드 : 20대의 비중이 가장 높음

fig = plot_population_pyramid(
    SPOP_2025_ADM_age_mid,
    gu_name='관악구',
    dong_name='대학동',
    hour=14
)

fig.show()
fig.write_html("html/2_4.html")


In [119]:
# 서울대학교가 있는 관악구 대학동, 오후 2시 생활인구 피라미드와 가장 유사한 행정동 Top 5를 비교
# 광진구 군자동(세종대학교), 광진구 화양동(건국대학교), 성동구 사근동(한양대학교), 성북구 안암동(고려대학교), 성북구 정릉3동(국민대학교)

fig, similar_dongs = cosine_similarity_pyramid(
    SPOP_2025_ADM_age_mid,
    gu_name='관악구',
    dong_name='대학동',
    hour=14,
    top_n=5
)

display(similar_dongs)

fig.show()
fig.write_html("html/2_5.html")


,SGG_NM,ADMI_NM,FULL_NM,ADMI_CD,COSINE_SIMILARITY
순위,,,,,
1,성북구,안암동,서울특별시 성북구 안암동,11290600,0.976916
2,광진구,군자동,서울특별시 광진구 군자동,11215730,0.976676
3,광진구,화양동,서울특별시 광진구 화양동,11215710,0.967082
4,성동구,사근동,서울특별시 성동구 사근동,11200550,0.965991
5,성북구,정릉3동,서울특별시 성북구 정릉3동,11290640,0.963662


In [120]:
# 서울의 대표적인 학군지인 강남구 대치1동, 오전 2시 생활인구 피라미드 : 10대와 부모세대인 40~50대 비중이 가장 높음
fig = plot_population_pyramid(
    SPOP_2025_ADM_age_mid,
    gu_name='강남구',
    dong_name='대치1동',
    hour=2
)

fig.show()
fig.write_html("html/2_6.html")


In [121]:
# 대치1동, 오전 2시 생활인구 피라미드와 가장 유사한 행정동 Top 5를 비교
# 서울의 대표적인 학군지인 고덕동, 중계동, 잠실동, 목동이 유사한 생활인구 피라미드 형태를 보임

fig, similar_dongs = wasserstein_distance_pyramid(
    SPOP_2025_ADM_age_mid,
    gu_name='강남구',
    dong_name='대치1동',
    hour=2,
    top_n=5
)

display(similar_dongs)

fig.show()
fig.write_html("html/2_7.html")


,SGG_NM,ADMI_NM,FULL_NM,ADMI_CD,WASSERSTEIN_DISTANCE
순위,,,,,
1,노원구,중계1동,서울특별시 노원구 중계1동,11350621,1.382624
2,송파구,잠실2동,서울특별시 송파구 잠실2동,11710670,1.570337
3,노원구,중계본동,서울특별시 노원구 중계본동,11350619,1.784491
4,강동구,고덕1동,서울특별시 강동구 고덕1동,11740550,1.861028
5,양천구,목1동,서울특별시 양천구 목1동,11470510,2.212960
